### Rag application using TypeSense 

In [12]:
import typesense
import os
from dotenv import load_dotenv
load_dotenv()
groq_api_key = os.getenv("GROQ_API_KEY")
typescript_api_key = os.getenv("TYPESCRIPT_API_KEY")




In [13]:
client = typesense.Client({
    'nodes': [{
        'host': 'a5u19mybkwc403h6p-1.a1.typesense.net',
        'port': 443,
        'protocol': 'https'
    }],
    'api_key': typescript_api_key,
    'connection_timeout_seconds': 2
})

In [14]:
books_schema = { 'name' : 'books' , 
                'fields' : [ {'name': 'title', 'type': 'string'}, 
                            {'name' : 'authors', 'type': 'string[]', 'facet' : True},
                            {'name': 'publication_year', 'type': 'int32', 'facet' : True},
                            {'name': 'ratings_count', 'type': 'int32'},
                            {'name' : 'average_rating', 'type' : 'float' } ],
                'default_sorting_field' : 'ratings_count' } 

print(client. collections.create(books_schema) )


{'created_at': 1790516731, 'curation_sets': [], 'default_sorting_field': 'ratings_count', 'enable_nested_fields': False, 'fields': [{'facet': False, 'index': True, 'infix': False, 'locale': '', 'name': 'title', 'optional': False, 'sort': False, 'stem': False, 'stem_dictionary': '', 'store': True, 'truncate_len': 100, 'type': 'string'}, {'facet': True, 'index': True, 'infix': False, 'locale': '', 'name': 'authors', 'optional': False, 'sort': False, 'stem': False, 'stem_dictionary': '', 'store': True, 'truncate_len': 100, 'type': 'string[]'}, {'facet': True, 'index': True, 'infix': False, 'locale': '', 'name': 'publication_year', 'optional': False, 'sort': True, 'stem': False, 'stem_dictionary': '', 'store': True, 'truncate_len': 100, 'type': 'int32'}, {'facet': False, 'index': True, 'infix': False, 'locale': '', 'name': 'ratings_count', 'optional': False, 'sort': True, 'stem': False, 'stem_dictionary': '', 'store': True, 'truncate_len': 100, 'type': 'int32'}, {'facet': False, 'index': T

In [24]:
with open('D:\\ProjectsGit\\ai-ml\\RAG\\data\\books.jsonl', 'r', encoding='utf-8') as json_file:
    data = json_file.read()
    client = typesense.Client({
        'nodes': [{
            'host': 'a5u19mybkwc403h6p-1.a1.typesense.net',
            'port': 443,
            'protocol': 'https'
        }],
        'api_key': typescript_api_key,
        'connection_timeout_seconds': 60,
        'num_retries': 5,
        'retry_interval_seconds': 2
    })

    results = []

    for start in range(0, len(data.splitlines()), 500):
        batch = '\n'.join(data.splitlines()[start:start + 500])
        result = client.collections['books'].documents.import_(
            batch,
            {'action': 'upsert'}
        )
        results.extend(result.splitlines() if isinstance(result, str) else result)

    print(f"Imported {len(results)} batches successfully.")
    

Imported 9979 batches successfully.


In [20]:
search_parameters = {
    'q' : "hary potter",
    'query_by' : 'title,authors',
    'sorted_by' : 'ratings_count:desc'
}
client.collections['books'].documents.search(search_parameters)

{'facet_counts': [],
 'found': 14,
 'hits': [{'document': {'authors': ['J.K. Rowling', ' Mary GrandPré'],
    'average_rating': 4.44,
    'id': '2',
    'image_url': 'https://images.gr-assets.com/books/1474154022m/3.jpg',
    'publication_year': 1997,
    'ratings_count': 4602479,
    'title': "Harry Potter and the Philosopher's Stone"},
   'highlight': {'authors': [{'matched_tokens': [], 'snippet': 'J.K. Rowling'},
     {'matched_tokens': ['Mary'], 'snippet': ' <mark>Mary</mark> GrandPré'}],
    'title': {'matched_tokens': ['Harry', 'Potter'],
     'snippet': "<mark>Harry</mark> <mark>Potter</mark> and the Philosopher's Stone"}},
   'highlights': [{'field': 'title',
     'matched_tokens': ['Harry', 'Potter'],
     'snippet': "<mark>Harry</mark> <mark>Potter</mark> and the Philosopher's Stone"},
    {'field': 'authors',
     'indices': [1],
     'matched_tokens': [['Mary']],
     'snippets': [' <mark>Mary</mark> GrandPré']}],
   'text_match': 1157451402721626233,
   'text_match_info': 

In [21]:
### Langchain integration with Typesense

from langchain_community.document_loaders import TextLoader
from langchain_community.vectorstores import Typesense
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_groq import ChatGroq


In [22]:
loader=TextLoader('../data/text_files/python_advanced.txt', encoding='utf-8')
documents=loader.load()
text_splitter=RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=0)
docs=text_splitter.split_documents(documents)

embeddings=HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2252.83it/s]


In [23]:
doc_search=Typesense.from_documents(docs, 
                                    embeddings, 
                                    typesense_client=client, 
                                    typesense_collection_name='lang-chain')

In [25]:
query="Machine Learning is broadly divided into three major types"
results=doc_search.similarity_search(query)
print(results[0].page_content)

ML is widely used in modern applications such as **recommendation systems, fraud detection, spam filtering, medical diagnosis, image recognition, speech processing, and predictive analytics**. Popular machine learning algorithms include Linear Regression, Logistic Regression, Decision Trees, Random Forest, Support Vector Machines, K-Nearest Neighbors, and Neural Networks.

The machine learning process generally involves **data collection, data preprocessing, feature engineering, model selection, training, evaluation, and deployment**. With the increasing availability of data and computational power, ML has become an import


In [26]:
retriever = doc_search.as_retriever(search_kwargs={"k": 2})
retriever


VectorStoreRetriever(tags=['Typesense', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.typesense.Typesense object at 0x00000249E4277110>, search_kwargs={'k': 2})

In [28]:
query = "What is the difference between a list and a tuple in Python?"
retriever.invoke(query)


[Document(metadata={'source': '../data/text_files/python_advanced.txt'}, page_content='# Introduction to Machine Learning\n\nMachine Learning (ML) is a branch of Artificial Intelligence (AI) that enables computers to learn from data and make predictions or decisions without being explicitly programmed for every task. Instead of following only fixed instructions, machine learning algorithms identify patterns and relationships within data and use them to solve problems or improve their performance over time.\n\nMachine Learning is broadly divided into three major types: **Supervised Learning, Unsupervised Learning, and Reinforcement Learning**. Supervised learning uses labeled data to train models for tasks such as classification and regression. Unsupervised learning works with unlabeled data to discover hidden patterns, groups, or structures. Reinforcement learning involves an agent learning through interactions with an environment by receiving rewards or penalties for its actions.'),
 